# Nados AI v1.1 — خادم الاستدلال السحابي

شغّل الخلية بالترتيب (Runtime → Run all). في النهاية ستحصل على **رابط عام** يستخدمه موقع Nados للنموذج المدرَّب.


In [ ]:
#@title 1) تثبيت llama.cpp وcloudflared
!wget -q https://github.com/ggml-org/llama.cpp/releases/download/b4458/llama-b4458-bin-ubuntu-x64.zip -O llama.zip
!unzip -q -o llama.zip -d llama-bin
!chmod +x llama-bin/llama-server
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared && chmod +x cloudflared
print('تم التثبيت ✓')


In [ ]:
#@title 2) تنزيل النموذج الأساسي والمحوّل المدرَّب
MODEL_SIZE = 'mini' #@param ['mini', 'full']
if MODEL_SIZE == 'mini':
    !wget -q https://huggingface.co/bartowski/gemma-2-2b-it-GGUF/resolve/main/gemma-2-2b-it-Q4_K_M.gguf -O base.gguf
    !wget -q https://huggingface.co/noore7xd/nados-models/resolve/main/nados-v1-1-mini-lora.gguf -O nados-lora.gguf
else:
    !wget -q https://huggingface.co/bartowski/gemma-2-9b-it-GGUF/resolve/main/gemma-2-9b-it-Q4_K_M.gguf -O base.gguf
    !wget -q https://huggingface.co/noore7xd/nados-models/resolve/main/nados-v1-1-lora.gguf -O nados-lora.gguf
import os
print('base:', round(os.path.getsize('base.gguf')/(1024**3), 2), 'GB | lora:', round(os.path.getsize('nados-lora.gguf')/(1024**2), 0), 'MB')


In [ ]:
#@title 3) تشغيل خادم Nados v1.1 + النفق العام
import subprocess, threading, time
server = subprocess.Popen(['llama-bin/llama-server', '-m', 'base.gguf', '--lora', 'nados-lora.gguf', '--host', '0.0.0.0', '--port', '8080', '--threads', '2'], stdout=open('server.log', 'w'), stderr=subprocess.STDOUT)
time.sleep(12)
print('خادم Nados v1.1 يعمل ✓')
tunnel = subprocess.Popen(['./cloudflared', 'tunnel', '--url', 'http://localhost:8080', '--no-autoupdate'], stdout=open('tunnel.log', 'w'), stderr=subprocess.STDOUT)
time.sleep(15)
log = open('tunnel.log').read()
import re
match = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', log)
if match:
    print('\n🌍 الرابط العام لنموذج Nados v1.1:')
    print(match.group(0))
    print('\nالصق هذا الرابط في NADOS_CLOUD_LLM_URL بملف .env أو في لوحة التدريب.')
else:
    print('لم يظهر الرابط بعد — أعد فتح tunnel.log')
